# Churn Prediction

This notebook will establishe the predictive baseline for `Beyond Churn`.

The goal is to answer:

> **Which customers are most likely to churn?**

I will build and evaluate churn-risk models before comparing predictive churn risk with causal treatment responsiveness in later notebooks.

This distinction is central to the project:

> **The customer most likely to churn is not necessarily the customer most likely to be saved.**

### Imports

In [1]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openml

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

### Data

In [2]:
# reload the dataset
dataset = openml.datasets.get_dataset(45580)

X, y, categorical_indicator, attribute_names = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute
)

df = X.copy()
df["y"] = y

df.shape

(11896, 180)

### Sanity Checks and Basic Data Preparation

In [3]:
# quick checks - see 01 notebook for more data integrity checks
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Churn rate: {df['y'].mean():.2%}")

df.head()

Rows: 11,896
Columns: 180
Churn rate: 3.43%


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,FACTOR11,FACTOR12,FACTOR13,FACTOR14,FACTOR15,FACTOR16,FACTOR17,FACTOR18,t,y
0,-3.064007,-1.272259,-4.075844,1.492514,2.328039,-0.330240,-0.770389,0.035541,2.439325,-0.847266,...,V2,V2,V5,V20,V9,V11,V1,V4,0,0
1,-4.574066,-3.541815,1.107371,0.447314,-0.471140,-0.567309,0.195963,0.383654,1.523472,-0.184596,...,V2,V3,V6,V20,V9,V15,V4,V4,0,0
2,-1.751471,-2.039692,-3.788823,0.624226,0.564614,-0.713385,1.003502,1.582819,-1.461687,0.844975,...,V5,V2,V11,V20,V9,V1,V4,V4,1,0
3,-2.030089,-0.235720,-5.711960,1.071841,2.441541,0.758484,-0.534295,1.614289,-1.022909,-1.383025,...,V2,V2,V4,V20,V11,V1,V4,V4,1,0
4,-2.377857,-2.478670,-1.051946,2.812768,2.019267,0.095542,-0.033324,-1.332319,4.316526,-0.537967,...,V2,V2,V7,V20,V2,V1,V2,V2,1,0


In [4]:
# define the target, treatment, and predictors
target_col = "y"
treatment_col = "t"

feature_cols = [
    col for col in df.columns
    if col not in [target_col, treatment_col]
]

X_model = df[feature_cols].copy()
y_model = df[target_col].copy()
treatment = df[treatment_col].copy()

### Important Note:
- I'm intentionally excluding `t` from the churn model. That is because `t` represents the intervention users received. But our model should describe customer churn risk from customer characteristics instead of learning whether a user was assigned to treatment and treatment might have affected their outcome. So, including treatment assignment would mix our predictive-risk concept with intervention effects. I will preserve treatment separately for later analyses.

In [5]:
# remove the constant feature we discovered in notebook 01
# drop FACTOR3 because it only contains one value - no meaningful predictive value
X_model = X_model.drop(columns="FACTOR3")

X_model.shape

(11896, 177)

In [6]:
# identify numeric and categorical predictors
categorical_cols = X_model.select_dtypes(
    include=["str", "category"]
).columns.tolist()

numeric_cols = X_model.select_dtypes(
    include=["number"]
).columns.tolist()

print(f"Numeric predictors: {len(numeric_cols)}")
print(f"Categorical predictors: {len(categorical_cols)}")
print(f"Total predictors: {X_model.shape[1]}")

Numeric predictors: 160
Categorical predictors: 17
Total predictors: 177


In [8]:
# double check missingness
print(f"Missing values: {X_model.isna().sum().sum()}")
print(f"Target classes:\n{y_model.value_counts()}")
assert X_model.isna().sum().sum() == 0
assert set(y_model.unique()) == {0, 1}

Missing values: 0
Target classes:
y
0    11488
1      408
Name: count, dtype: int64
